# JupyterLite wasm client — local debug notebook

This notebook exercises the OpenGRIS Scaler wasm client from the browser.
It is **not** part of the published docs — it is for local debugging only.

**Prerequisites (run `scripts/test_jupyterlite.sh` first):**
- Object storage server running on `tcp://127.0.0.1:7379`
- Scheduler running on `ws://127.0.0.1:7380`
- At least one worker connected to the scheduler
- Wasm wheel deployed to `_static/wasm/` (served by this HTTP server)

Run cells top-to-bottom.  Re-run **Cell 1** (install) only once per page load.

In [ ]:
# Cell 1 — install the wasm wheel via micropip.
# Runs only in Pyodide (sys.platform == 'emscripten'); safe to re-run.
import sys
import js  # noqa: F401  # available in Pyodide/JupyterLite

assert sys.platform == 'emscripten', 'This notebook must run inside JupyterLite (Pyodide kernel)'

import micropip

# cloudpickle is a pure-Python dependency not bundled by Pyodide.
await micropip.install('cloudpickle')

# Pyodide 0.29.x bundles tblib 3.0.0, but the native worker uses tblib >= 3.2.0
# which pickles exceptions via 'unpickle_exception_with_attrs' (added in 3.2.0).
# Install a matching version into the browser so remote-task exceptions can be
# unpickled here without AttributeError.
await micropip.install('tblib>=3.2.0')

# Install the scaler wasm wheel from the static files served alongside this site.
_wheel_base = js.location.origin + '/_static/wasm/'

# List what is available so we can pick the right filename.
import pyodide.http
_listing = await pyodide.http.pyfetch(_wheel_base)
_html = await _listing.string()

import re
_whl = next(
    m.group(0)
    for m in re.finditer(r'opengris_scaler[^"]*wasm32\.whl', _html)
)

# Cache-bust the wheel URL: across local rebuilds the version often does not
# change, so the same URL would otherwise be served from the browser /
# service-worker cache and micropip would re-install the stale wheel.
_wheel_url = _wheel_base + _whl + f'?t={int(js.Date.now())}'
print(f'Installing: {_wheel_url}')
await micropip.install(_wheel_url)
print('Done.')


In [ ]:
# Cell 2 — import smoke-check.
from scaler.io.ymq._ymq_wasm import ConnectorSocket, AddressType
from scaler import Client

print(f'sys.platform     : {sys.platform}')
print(f'ConnectorSocket  : {ConnectorSocket}')
print(f'AddressType      : {list(AddressType)}')
print(f'Client           : {Client}')
print('imports OK')

In [ ]:
# Cell 3 — connection settings.
# Edit SCHEDULER_ADDRESS to match the address printed by scripts/test_jupyterlite.sh.
SCHEDULER_ADDRESS = 'ws://127.0.0.1:7380'
OBJECT_STORAGE_ADDRESS = None  # use whatever the scheduler advertises


In [ ]:
# Cell 4 — simple math.sqrt batch.
import math

with Client(address=SCHEDULER_ADDRESS, object_storage_address=OBJECT_STORAGE_ADDRESS) as client:
    futures = [client.submit(math.sqrt, value) for value in range(16)]
    results = [f.result() for f in futures]

for i, r in enumerate(results):
    print(f'sqrt({i:2d}) = {r:.6f}')

In [ ]:
# Cell 5 — numpy array task.
# numpy is bundled in Pyodide; no micropip install needed.
import numpy as np


def numpy_task(n: int) -> dict:
    """Create a random (n, n) matrix, return its trace and Frobenius norm."""
    rng = np.random.default_rng(seed=n)
    mat = rng.standard_normal((n, n))
    return {'n': n, 'trace': float(np.trace(mat)), 'frobenius': float(np.linalg.norm(mat))}


with Client(address=SCHEDULER_ADDRESS, object_storage_address=OBJECT_STORAGE_ADDRESS) as client:
    futures = [client.submit(numpy_task, n) for n in [10, 20, 50, 100]]
    results = [f.result() for f in futures]

for r in results:
    print(f"n={r['n']:3d}  trace={r['trace']:+8.3f}  ‖A‖_F={r['frobenius']:.3f}")

In [ ]:
# Cell 6 — pandas DataFrame task.
# pandas is bundled in Pyodide; no micropip install needed.
import pandas as pd


def pandas_task(ticker: str, days: int) -> dict:
    """Simulate a price series, return summary statistics as a dict."""
    rng = np.random.default_rng(seed=abs(hash(ticker)) % (2**31))
    log_returns = rng.normal(0.0, 0.01, size=days)
    prices = pd.Series(100.0 * np.exp(np.cumsum(log_returns)), name=ticker)
    return prices.describe().to_dict()


tickers = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA']

with Client(address=SCHEDULER_ADDRESS, object_storage_address=OBJECT_STORAGE_ADDRESS) as client:
    futures = {t: client.submit(pandas_task, t, 252) for t in tickers}
    results = {t: f.result() for t, f in futures.items()}

summary = pd.DataFrame(results).T
print(summary[['mean', 'std', 'min', 'max']].round(3).to_string())

In [ ]:
# Cell 7 — round-trip latency micro-benchmark.
# Submits 100 trivial tasks one at a time and measures round-trip time.
import time


def identity(x):
    return x


N = 100
with Client(address=SCHEDULER_ADDRESS, object_storage_address=OBJECT_STORAGE_ADDRESS) as client:
    t0 = time.monotonic()
    for i in range(N):
        client.submit(identity, i).result()
    elapsed = time.monotonic() - t0

print(f'{N} sequential tasks in {elapsed:.2f}s  ({elapsed / N * 1000:.1f} ms/task)')